# Logistic Regression - Complete Tutorial

**Goal**: Predict a categorical outcome (binary or multiclass) using probability-based classification.

**Contents:**
- Part 1: Binary logistic regression fundamentals
- Part 2: All types (Multiclass, Regularized, Decision Boundaries, ROC/AUC)

---

# PART 1: Binary Logistic Regression Fundamentals

---

## 1. Conceptual Overview

### What is Classification?

Classification predicts which **category** a data point belongs to.

- **Binary Classification**: 2 classes (Pass/Fail, Yes/No, 0/1)
- **Multiclass Classification**: 3+ classes (Cat/Dog/Bird)

### Why Not Use Linear Regression for Binary Outcomes?

**Problem**: Linear regression can predict values outside [0,1] range, like -0.5 or 1.7, which don't make sense as probabilities.

**Solution**: Use logistic regression, which outputs values between 0 and 1 (probabilities).

### The Sigmoid Function

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

This function:
- Takes any input z (from -∞ to +∞)
- Outputs values between 0 and 1
- Creates an S-shaped curve

### When to Use Logistic Regression
- Target variable is **binary** (two categories)
- You want **probability** of belonging to a class
- Features have some linear relationship with log-odds

---

## 2. Setup and Imports

In [ ]:
# Import essential libraries
import numpy as np  # for numerical operations
import pandas as pd  # for data manipulation
import matplotlib.pyplot as plt  # for visualization

# Set display options
np.set_printoptions(precision=4)
pd.set_option('display.precision', 4)

## 3. Visualize the Sigmoid Function

In [ ]:
# Define the sigmoid function
def sigmoid(z):
    """Sigmoid function: σ(z) = 1 / (1 + e^(-z))"""
    return 1 / (1 + np.exp(-z))

# Create z values from -10 to 10
z_values = np.linspace(-10, 10, 100)

# Calculate sigmoid for each z
sigmoid_values = sigmoid(z_values)

# Plot the sigmoid curve
plt.figure(figsize=(10, 5))
plt.plot(z_values, sigmoid_values, 'b-', linewidth=2, label='Sigmoid: σ(z) = 1/(1+e⁻ᶻ)')
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Decision boundary (0.5)')
plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
plt.axhline(y=1, color='gray', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
plt.xlabel('z = β₀ + β₁x', fontsize=12)
plt.ylabel('σ(z) = P(y=1)', fontsize=12)
plt.title('The Sigmoid Function', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.ylim(-0.1, 1.1)
plt.show()

# Note: Output is always between 0 and 1 → perfect for probability!

In [ ]:
# Demonstrate sigmoid values at key points
demo_z = [-5, -2, 0, 2, 5]
demo_sigmoid = [sigmoid(z) for z in demo_z]

pd.DataFrame({'z': demo_z, 'σ(z)': demo_sigmoid})

# When z = 0, σ(z) = 0.5 (equally likely)
# When z is very negative, σ(z) → 0
# When z is very positive, σ(z) → 1

## 4. Create Synthetic Dataset

**Scenario**: Predict if a student passes (1) or fails (0) based on hours studied.

- X = Hours of study
- y = Pass (1) or Fail (0)

In [ ]:
# Create synthetic dataset (20 students)
# Pattern: More study hours → higher chance of passing

X = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0,
              5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0])

y = np.array([0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
              0, 1, 1, 1, 1, 1, 1, 1, 1, 1])

# Create DataFrame
df = pd.DataFrame({'Hours_Studied': X, 'Pass': y})
df['Result'] = df['Pass'].map({0: 'Fail', 1: 'Pass'})  # add readable labels
df

In [ ]:
# Visualize the data
plt.figure(figsize=(10, 5))

# Plot pass and fail separately with different colors
fail_mask = y == 0
pass_mask = y == 1

plt.scatter(X[fail_mask], y[fail_mask], color='red', s=100, 
            edgecolor='black', label='Fail (0)', zorder=5)
plt.scatter(X[pass_mask], y[pass_mask], color='green', s=100, 
            edgecolor='black', label='Pass (1)', zorder=5)

plt.xlabel('Hours Studied', fontsize=12)
plt.ylabel('Pass (1) / Fail (0)', fontsize=12)
plt.title('Study Hours vs Pass/Fail Outcome', fontsize=14)
plt.legend(fontsize=10)
plt.yticks([0, 1], ['Fail (0)', 'Pass (1)'])
plt.grid(True, alpha=0.3)
plt.show()

# Observation: Students who study more tend to pass

## 5. The Logistic Regression Equation

### The Model

$$P(y=1|x) = \sigma(\beta_0 + \beta_1 x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x)}}$$

**Symbol meanings:**

| Symbol | Name | Meaning |
|--------|------|--------|
| $P(y=1|x)$ | Probability | Probability of y=1 given x |
| $\sigma$ | Sigmoid | Function that converts to probability |
| $\beta_0$ | Intercept | Baseline log-odds when x=0 |
| $\beta_1$ | Slope | How much log-odds changes per unit x |
| $x$ | Feature | Input variable (hours studied) |

### Understanding Log-Odds

**Odds**: $\frac{p}{1-p}$ = probability of success / probability of failure

**Log-odds (logit)**: $\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x$

Logistic regression is linear in log-odds, not in probability.

---

## 6. Log-Odds and Probability Relationship

In [ ]:
# Show relationship between probability, odds, and log-odds
probabilities = np.array([0.1, 0.25, 0.5, 0.75, 0.9])
odds = probabilities / (1 - probabilities)
log_odds = np.log(odds)

pd.DataFrame({
    'Probability (p)': probabilities,
    'Odds (p/(1-p))': np.round(odds, 3),
    'Log-Odds (ln(odds))': np.round(log_odds, 3)
})

# Note:
# - p = 0.5 → odds = 1 → log-odds = 0 (equally likely)
# - p > 0.5 → odds > 1 → log-odds > 0 (more likely to pass)
# - p < 0.5 → odds < 1 → log-odds < 0 (more likely to fail)

## 7. How Logistic Regression Learns (Concept)

### The Loss Function (Binary Cross-Entropy)

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\left[ y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i) \right]$$

**In simple terms:**
- When actual y = 1: we want predicted probability $\hat{p}$ to be close to 1
- When actual y = 0: we want predicted probability $\hat{p}$ to be close to 0
- This loss penalizes wrong predictions

### How Parameters are Found

Unlike linear regression (closed-form solution), logistic regression uses **iterative optimization**:

1. Start with initial guesses for $\beta_0$ and $\beta_1$
2. Calculate predictions and loss
3. Adjust parameters to reduce loss (gradient descent)
4. Repeat until loss stops decreasing

**Gradient Update (simplified):**
$$\beta_{new} = \beta_{old} - \alpha \cdot \frac{\partial \mathcal{L}}{\partial \beta}$$

Where $\alpha$ = learning rate (step size)

---

In [ ]:
# Demonstrate the loss function
def binary_cross_entropy(y_true, y_pred):
    """Calculate binary cross-entropy loss."""
    # Clip predictions to avoid log(0)
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Example: Compare loss for good vs bad predictions
y_actual = np.array([1, 0, 1, 1])  # actual labels

# Good predictions (close to actual)
y_pred_good = np.array([0.9, 0.1, 0.8, 0.95])

# Bad predictions (far from actual)
y_pred_bad = np.array([0.2, 0.8, 0.3, 0.4])

print("Actual labels:", y_actual)
print("\nGood predictions:", y_pred_good)
print(f"Loss (good): {binary_cross_entropy(y_actual, y_pred_good):.4f}")
print("\nBad predictions:", y_pred_bad)
print(f"Loss (bad): {binary_cross_entropy(y_actual, y_pred_bad):.4f}")
print("\n→ Lower loss = better predictions")

## 8. Fit Model Using Scikit-Learn

In [ ]:
# Import LogisticRegression
from sklearn.linear_model import LogisticRegression

# Reshape X for sklearn (needs 2D array)
X_reshaped = X.reshape(-1, 1)

# Create and fit the model
model = LogisticRegression()
model.fit(X_reshaped, y)

# Get coefficients
beta_0 = model.intercept_[0]  # intercept
beta_1 = model.coef_[0][0]    # slope

print("MODEL COEFFICIENTS")
print("=" * 40)
print(f"β₀ (Intercept): {beta_0:.4f}")
print(f"β₁ (Slope):     {beta_1:.4f}")
print("=" * 40)

In [ ]:
# Interpretation of coefficients
print("INTERPRETATION")
print(f"\nβ₁ = {beta_1:.4f}")
print(f"→ Each additional hour of study increases log-odds by {beta_1:.4f}")
print(f"→ In terms of odds: multiplied by e^{beta_1:.4f} = {np.exp(beta_1):.4f}")
print(f"\nThis means each extra hour increases odds of passing by {(np.exp(beta_1)-1)*100:.1f}%")

## 9. Get Predictions

In [ ]:
# Get predicted probabilities
y_prob = model.predict_proba(X_reshaped)[:, 1]  # probability of class 1 (Pass)

# Get class predictions (using threshold = 0.5)
y_pred = model.predict(X_reshaped)  # 0 or 1

# Show results
results_df = pd.DataFrame({
    'Hours': X,
    'Actual': y,
    'Prob(Pass)': np.round(y_prob, 3),
    'Predicted': y_pred,
    'Correct?': y == y_pred
})
results_df

In [ ]:
# How prediction works:
# 1. Calculate z = β₀ + β₁ × x
# 2. Calculate probability = sigmoid(z)
# 3. If probability >= 0.5, predict 1 (Pass), else predict 0 (Fail)

# Manual example for student who studied 5 hours
hours = 5
z = beta_0 + beta_1 * hours
prob = sigmoid(z)
pred = 1 if prob >= 0.5 else 0

print(f"For {hours} hours of study:")
print(f"  z = {beta_0:.4f} + {beta_1:.4f} × {hours} = {z:.4f}")
print(f"  P(Pass) = sigmoid({z:.4f}) = {prob:.4f}")
print(f"  Since {prob:.4f} >= 0.5, predict: {'Pass' if pred == 1 else 'Fail'}")

## 10. Visualize the Logistic Curve

In [ ]:
# Create smooth curve for visualization
X_smooth = np.linspace(0, 11, 100)
z_smooth = beta_0 + beta_1 * X_smooth
prob_smooth = sigmoid(z_smooth)

# Plot
plt.figure(figsize=(10, 6))

# Plot actual data points
plt.scatter(X[fail_mask], y[fail_mask], color='red', s=100, 
            edgecolor='black', label='Actual: Fail', zorder=5)
plt.scatter(X[pass_mask], y[pass_mask], color='green', s=100, 
            edgecolor='black', label='Actual: Pass', zorder=5)

# Plot logistic curve
plt.plot(X_smooth, prob_smooth, 'b-', linewidth=2, 
         label=f'P(Pass) = σ({beta_0:.2f} + {beta_1:.2f}x)')

# Decision boundary line
plt.axhline(y=0.5, color='orange', linestyle='--', alpha=0.7, 
            label='Decision boundary (p=0.5)')

# Find x where probability = 0.5 (decision boundary)
x_boundary = -beta_0 / beta_1
plt.axvline(x=x_boundary, color='purple', linestyle=':', alpha=0.7,
            label=f'Boundary at x={x_boundary:.2f}')

plt.xlabel('Hours Studied', fontsize=12)
plt.ylabel('Probability of Passing', fontsize=12)
plt.title('Logistic Regression: Predicting Pass/Fail', fontsize=14)
plt.legend(loc='right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.ylim(-0.1, 1.1)
plt.show()

print(f"Decision boundary: Students who study more than {x_boundary:.2f} hours are predicted to pass")

## 11. Confusion Matrix

### What is a Confusion Matrix?

A table showing how predictions compare to actual values:

|  | Predicted: 0 (Fail) | Predicted: 1 (Pass) |
|--|---------------------|---------------------|
| **Actual: 0 (Fail)** | TN (True Negative) | FP (False Positive) |
| **Actual: 1 (Pass)** | FN (False Negative) | TP (True Positive) |

**Terms:**
- **TP**: Correctly predicted Pass
- **TN**: Correctly predicted Fail
- **FP**: Predicted Pass, but actually Fail (Type I error)
- **FN**: Predicted Fail, but actually Pass (Type II error)

In [ ]:
# Create confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y, y_pred)

print("CONFUSION MATRIX")
print("=" * 30)
print(f"             Predicted")
print(f"             Fail  Pass")
print(f"Actual Fail   {cm[0,0]}     {cm[0,1]}")
print(f"Actual Pass   {cm[1,0]}     {cm[1,1]}")
print("=" * 30)

# Extract values
TN, FP, FN, TP = cm.ravel()
print(f"\nTrue Negatives (TN):  {TN}")
print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")
print(f"True Positives (TP):  {TP}")

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fail', 'Pass'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix', fontsize=14)
plt.show()

## 12. Evaluation Metrics

### Key Metrics with Formulas

**1. Accuracy** - Overall correctness
$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

**2. Precision** - Of predicted positives, how many are correct?
$$Precision = \frac{TP}{TP + FP}$$

**3. Recall (Sensitivity)** - Of actual positives, how many did we catch?
$$Recall = \frac{TP}{TP + FN}$$

**4. F1-Score** - Harmonic mean of Precision and Recall
$$F1 = \frac{2 \times Precision \times Recall}{Precision + Recall} = \frac{2TP}{2TP + FP + FN}$$

---

In [ ]:
# Calculate metrics manually
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("EVALUATION METRICS (Manual Calculation)")
print("=" * 45)
print(f"Accuracy  = (TP+TN)/(TP+TN+FP+FN) = ({TP}+{TN})/({TP}+{TN}+{FP}+{FN}) = {accuracy:.4f}")
print(f"Precision = TP/(TP+FP) = {TP}/({TP}+{FP}) = {precision:.4f}")
print(f"Recall    = TP/(TP+FN) = {TP}/({TP}+{FN}) = {recall:.4f}")
print(f"F1-Score  = 2×P×R/(P+R) = {f1:.4f}")
print("=" * 45)

In [ ]:
# Verify with sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print("VERIFICATION WITH SKLEARN")
print(f"Accuracy:  {accuracy_score(y, y_pred):.4f}")
print(f"Precision: {precision_score(y, y_pred):.4f}")
print(f"Recall:    {recall_score(y, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y, y_pred):.4f}")

In [ ]:
# Full classification report
print("\nFULL CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(y, y_pred, target_names=['Fail', 'Pass']))

## 13. Interpretation of Metrics

| Metric | Meaning | When to Focus On |
|--------|---------|------------------|
| **Accuracy** | Overall % correct | Balanced classes |
| **Precision** | "When I predict Pass, how often am I right?" | Cost of false positives is high |
| **Recall** | "Of all actual Pass students, how many did I catch?" | Missing positives is costly |
| **F1-Score** | Balance between Precision and Recall | Imbalanced classes |

### Our Model's Performance

- High accuracy → model is generally correct
- Check precision vs recall to understand error types

## 14. Changing the Threshold

In [ ]:
# Default threshold is 0.5, but we can change it
# Lower threshold → more Pass predictions (higher recall, lower precision)
# Higher threshold → fewer Pass predictions (lower recall, higher precision)

thresholds = [0.3, 0.5, 0.7]

print("EFFECT OF DIFFERENT THRESHOLDS")
print("=" * 50)

for thresh in thresholds:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    acc = accuracy_score(y, y_pred_thresh)
    prec = precision_score(y, y_pred_thresh, zero_division=0)
    rec = recall_score(y, y_pred_thresh, zero_division=0)
    
    print(f"\nThreshold = {thresh}")
    print(f"  Predictions: {sum(y_pred_thresh)} Pass, {len(y_pred_thresh)-sum(y_pred_thresh)} Fail")
    print(f"  Accuracy: {acc:.3f}, Precision: {prec:.3f}, Recall: {rec:.3f}")

## 15. Making New Predictions

In [ ]:
# Predict for new students
new_students = np.array([2, 4, 6, 8]).reshape(-1, 1)
new_probs = model.predict_proba(new_students)[:, 1]
new_preds = model.predict(new_students)

pd.DataFrame({
    'Hours Studied': new_students.flatten(),
    'P(Pass)': np.round(new_probs, 3),
    'Prediction': ['Pass' if p == 1 else 'Fail' for p in new_preds]
})

## 16. Summary

### Key Takeaways

1. **Logistic regression** predicts probability of belonging to a class

2. Uses the **sigmoid function** to convert linear combination to probability:
   $$P(y=1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x)}}$$

3. **Coefficients ($\beta$)** are in terms of log-odds, not probability

4. **Threshold** (default 0.5) converts probability to class prediction

5. **Confusion matrix** shows TP, TN, FP, FN

6. **Key metrics**:
   - Accuracy = overall correctness
   - Precision = correctness of positive predictions
   - Recall = coverage of actual positives

### Formulas Cheat Sheet

| Formula | Purpose |
|---------|--------|
| $\sigma(z) = \frac{1}{1 + e^{-z}}$ | Sigmoid function |
| $Accuracy = \frac{TP + TN}{Total}$ | Overall correctness |
| $Precision = \frac{TP}{TP + FP}$ | Positive prediction accuracy |
| $Recall = \frac{TP}{TP + FN}$ | Positive detection rate |
| $F1 = \frac{2 \times P \times R}{P + R}$ | Harmonic mean of P and R |

---
**End of Logistic Regression Tutorial**

---

# PART 2: Types of Logistic Regression

---

## 17. Binary Logistic Regression (Review)

### Formula
$$P(y=1|x) = \sigma(\beta_0 + \beta_1 x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x)}}$$

### Log-Likelihood Function (What We Maximize)
$$\ell(\boldsymbol{\beta}) = \sum_{i=1}^{n}\left[ y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i) \right]$$

This is the **maximum likelihood estimator (MLE)** - we find β that maximizes this.

### What Makes It Different
- Output is **binary** (0 or 1)
- Predicts **probability** of class 1
- Uses **sigmoid** to bound output [0, 1]

### When to Use
- Binary classification (yes/no, pass/fail, spam/not spam)
- Need probability estimates
- Features have linear relationship with log-odds

In [ ]:
# Binary Logistic Regression - Full statsmodels summary
import statsmodels.api as sm

# Prepare data with constant
X_binary = X.reshape(-1, 1)
X_binary_const = sm.add_constant(X_binary)

# Fit Logit model
logit_model = sm.Logit(y, X_binary_const).fit(disp=0)
print(logit_model.summary())

In [ ]:
# Interpretation of Logit model
print("BINARY LOGISTIC REGRESSION INTERPRETATION")
print("=" * 55)
print(f"\nCoefficients (in log-odds):")
print(f"  β₀ (intercept) = {logit_model.params[0]:.4f}")
print(f"  β₁ (hours) = {logit_model.params[1]:.4f}")
print(f"\nOdds Ratio Interpretation:")
print(f"  exp(β₁) = {np.exp(logit_model.params[1]):.4f}")
print(f"  → Each extra hour multiplies odds of passing by {np.exp(logit_model.params[1]):.2f}")
print(f"\nModel Fit:")
print(f"  Pseudo R² (McFadden) = {logit_model.prsquared:.4f}")
print(f"  Log-Likelihood = {logit_model.llf:.4f}")

---

## 18. Multiclass Logistic Regression (One-vs-Rest)

### The Problem
Binary logistic regression handles 2 classes. What about 3+ classes?

### One-vs-Rest (OvR) Strategy
For K classes, train K binary classifiers:
- Classifier 1: Class A vs (B, C, D, ...)
- Classifier 2: Class B vs (A, C, D, ...)
- ...

### Softmax Function (for Multinomial)
$$P(y=k|\mathbf{x}) = \frac{e^{\boldsymbol{\beta}_k^T \mathbf{x}}}{\sum_{j=1}^{K} e^{\boldsymbol{\beta}_j^T \mathbf{x}}}$$

This ensures all class probabilities sum to 1.

### When to Use
- Target has 3+ categories (e.g., Low/Medium/High, Cat/Dog/Bird)
- No natural ordering (use ordinal regression if ordered)

In [ ]:
# Multiclass Logistic Regression - Synthetic Dataset
# Scenario: Predict grade (A, B, C) based on hours studied and previous score

np.random.seed(42)
n = 30

# Features
hours_multi = np.random.uniform(1, 10, n)
prev_score = np.random.uniform(40, 100, n)

# Create labels based on combined score
combined = 0.6 * hours_multi + 0.4 * (prev_score / 10)
grade = np.where(combined > 7, 'A', np.where(combined > 5, 'B', 'C'))

df_multi = pd.DataFrame({
    'Hours_Studied': np.round(hours_multi, 1),
    'Previous_Score': np.round(prev_score, 1),
    'Grade': grade
})
df_multi.head(10)

In [ ]:
# Fit Multiclass Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
y_multi = le.fit_transform(df_multi['Grade'])  # A=0, B=1, C=2

X_multi = df_multi[['Hours_Studied', 'Previous_Score']].values

# One-vs-Rest (default for multi_class='auto' with >2 classes)
multi_model = LogisticRegression(multi_class='ovr', max_iter=1000)
multi_model.fit(X_multi, y_multi)

# Predictions
y_pred_multi = multi_model.predict(X_multi)
y_prob_multi = multi_model.predict_proba(X_multi)

# Show probabilities for first 5 samples
prob_df = pd.DataFrame(y_prob_multi, columns=['P(A)', 'P(B)', 'P(C)'])
prob_df['Predicted'] = le.inverse_transform(y_pred_multi)
prob_df['Actual'] = df_multi['Grade'].values
prob_df.head()

In [ ]:
# Multiclass Confusion Matrix and Metrics
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

cm_multi = confusion_matrix(y_multi, y_pred_multi)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_multi, display_labels=['A', 'B', 'C'])
disp.plot(cmap='Blues')
plt.title('Multiclass Confusion Matrix', fontsize=14)
plt.show()

print("\nCLASSIFICATION REPORT")
print(classification_report(y_multi, y_pred_multi, target_names=['A', 'B', 'C']))

---

## 19. Regularized Logistic Regression

### Why Regularization?
- Prevents **overfitting** when many features
- Handles **multicollinearity**
- Improves **generalization** to new data

### L2 Regularization (Ridge)

$$\mathcal{L}_{L2} = -\sum_{i=1}^{n}\left[ y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i) \right] + \lambda \sum_{j=1}^{p}\beta_j^2$$

- Shrinks coefficients toward zero
- Never sets them exactly to zero
- Default in sklearn LogisticRegression

### L1 Regularization (Lasso)

$$\mathcal{L}_{L1} = -\sum_{i=1}^{n}\left[ y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i) \right] + \lambda \sum_{j=1}^{p}|\beta_j|$$

- Can set coefficients exactly to zero → **feature selection**
- Produces sparse models

In [ ]:
# Compare L1 vs L2 Regularization
from sklearn.preprocessing import StandardScaler

# Create dataset with more features (some irrelevant)
np.random.seed(42)
n_samples = 50

# Relevant features
hours_reg = np.random.uniform(1, 10, n_samples)
prev_score_reg = np.random.uniform(40, 100, n_samples)

# Irrelevant features (noise)
noise1 = np.random.normal(0, 1, n_samples)
noise2 = np.random.normal(0, 1, n_samples)
noise3 = np.random.normal(0, 1, n_samples)

# Target
y_reg = (0.5 * hours_reg + 0.3 * (prev_score_reg/10) + np.random.normal(0, 0.5, n_samples) > 5).astype(int)

X_reg = np.column_stack([hours_reg, prev_score_reg, noise1, noise2, noise3])
X_reg_scaled = StandardScaler().fit_transform(X_reg)

feature_names = ['Hours', 'PrevScore', 'Noise1', 'Noise2', 'Noise3']

In [ ]:
# Fit models with different regularization
# Note: sklearn uses C = 1/λ, so smaller C = stronger regularization

# No regularization (very large C)
model_no_reg = LogisticRegression(penalty=None, max_iter=1000).fit(X_reg_scaled, y_reg)

# L2 regularization (Ridge)
model_l2 = LogisticRegression(penalty='l2', C=0.1, max_iter=1000).fit(X_reg_scaled, y_reg)

# L1 regularization (Lasso)
model_l1 = LogisticRegression(penalty='l1', C=0.1, solver='saga', max_iter=1000).fit(X_reg_scaled, y_reg)

# Compare coefficients
coef_comparison = pd.DataFrame({
    'Feature': feature_names,
    'No Reg': np.round(model_no_reg.coef_[0], 4),
    'L2 (Ridge)': np.round(model_l2.coef_[0], 4),
    'L1 (Lasso)': np.round(model_l1.coef_[0], 4)
})
coef_comparison

# Notice: L1 tends to set noise coefficients closer to or exactly zero

In [ ]:
# Visualize coefficient comparison
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(feature_names))
width = 0.25

ax.bar(x_pos - width, model_no_reg.coef_[0], width, label='No Reg', color='blue', alpha=0.7)
ax.bar(x_pos, model_l2.coef_[0], width, label='L2 (Ridge)', color='green', alpha=0.7)
ax.bar(x_pos + width, model_l1.coef_[0], width, label='L1 (Lasso)', color='red', alpha=0.7)

ax.set_xlabel('Feature')
ax.set_ylabel('Coefficient Value')
ax.set_title('Regularization Effect on Coefficients')
ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names)
ax.legend()
ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax.grid(True, alpha=0.3)
plt.show()

print("→ L1 (Lasso) tends to eliminate irrelevant features (Noise) by setting coefficients to zero")

---

## 20. Decision Boundary Visualization

### What is a Decision Boundary?
The line (or surface) where $P(y=1) = 0.5$, i.e., where the model is equally likely to predict either class.

### For Logistic Regression:
$$\beta_0 + \beta_1 x_1 + \beta_2 x_2 = 0$$

Solving for $x_2$:
$$x_2 = -\frac{\beta_0}{\beta_2} - \frac{\beta_1}{\beta_2} x_1$$

This is a **linear boundary** (straight line in 2D).

In [ ]:
# Create 2D dataset for decision boundary visualization
np.random.seed(42)
n = 100

# Two features
x1 = np.random.uniform(0, 10, n)
x2 = np.random.uniform(0, 10, n)

# Binary target: pass if x1 + x2 > 10 (with noise)
y_boundary = ((x1 + x2 + np.random.normal(0, 1, n)) > 10).astype(int)

X_boundary = np.column_stack([x1, x2])

# Fit logistic regression
boundary_model = LogisticRegression()
boundary_model.fit(X_boundary, y_boundary)

print(f"Coefficients: β₀={boundary_model.intercept_[0]:.4f}, β₁={boundary_model.coef_[0][0]:.4f}, β₂={boundary_model.coef_[0][1]:.4f}")

In [ ]:
# Plot decision boundary
plt.figure(figsize=(10, 8))

# Create mesh grid for contour
xx, yy = np.meshgrid(np.linspace(0, 10, 200), np.linspace(0, 10, 200))
Z = boundary_model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
Z = Z.reshape(xx.shape)

# Plot probability contours
contour = plt.contourf(xx, yy, Z, levels=np.linspace(0, 1, 11), cmap='RdYlBu', alpha=0.7)
plt.colorbar(contour, label='P(Pass)')

# Plot decision boundary (where P = 0.5)
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2, linestyles='--')

# Plot data points
scatter = plt.scatter(x1, x2, c=y_boundary, cmap='RdYlBu', edgecolor='black', s=80)
plt.xlabel('Feature 1 (x₁)', fontsize=12)
plt.ylabel('Feature 2 (x₂)', fontsize=12)
plt.title('Logistic Regression Decision Boundary', fontsize=14)
plt.legend(*scatter.legend_elements(), title='Class', loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()

print("Black dashed line = Decision boundary where P(y=1) = 0.5")

---

## 21. Ordinal Logistic Regression (Concept)

### When to Use
When target variable has **ordered categories**:
- Low < Medium < High
- Poor < Fair < Good < Excellent
- 1 star < 2 stars < 3 stars < 4 stars < 5 stars

### Formula (Cumulative Logit Model)
$$\log\left(\frac{P(Y \leq k)}{P(Y > k)}\right) = \alpha_k - \boldsymbol{\beta}^T \mathbf{x}$$

Where:
- $\alpha_k$ = threshold for category k
- Single set of β coefficients (parallel lines assumption)

### Key Difference from Multinomial
- Uses ordering information
- More efficient (fewer parameters)
- Assumes proportional odds

*Note: sklearn doesn't have built-in ordinal regression. Use `mord` or `statsmodels` for full implementation.*

---

## 22. ROC Curve and AUC

### What is ROC Curve?
ROC = **Receiver Operating Characteristic**
- Plots **True Positive Rate (TPR)** vs **False Positive Rate (FPR)** at different thresholds

### Formulas
$$TPR = \frac{TP}{TP + FN} = Recall$$

$$FPR = \frac{FP}{FP + TN}$$

### AUC (Area Under Curve)
- AUC = 0.5: Random classifier (no skill)
- AUC = 1.0: Perfect classifier
- AUC > 0.7: Acceptable
- AUC > 0.8: Good
- AUC > 0.9: Excellent

In [ ]:
# ROC Curve using our original binary data
from sklearn.metrics import roc_curve, roc_auc_score

# Get probabilities from original model
y_prob_roc = model.predict_proba(X_reshaped)[:, 1]

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y, y_prob_roc)
auc_score = roc_auc_score(y, y_prob_roc)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Classifier (AUC = 0.5)')
plt.fill_between(fpr, tpr, alpha=0.3)
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('ROC Curve', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

print(f"AUC Score: {auc_score:.4f}")

---

## 23. Complete Comparison Summary

| Type | Classes | Regularization | Key Feature | Use Case |
|------|---------|----------------|-------------|----------|
| **Binary** | 2 | None | Sigmoid output | Pass/Fail, Yes/No |
| **Multinomial (OvR)** | 3+ | None | Softmax output | Unordered categories |
| **Ridge (L2)** | Any | L2 penalty | Shrinks coefficients | Multicollinearity |
| **Lasso (L1)** | Any | L1 penalty | Feature selection | Many irrelevant features |
| **Ordinal** | 3+ ordered | None | Cumulative logits | Ordered categories |

### Key Formulas Reference

| Formula | Name | Purpose |
|---------|------|---------|
| $\sigma(z) = \frac{1}{1+e^{-z}}$ | Sigmoid | Converts to probability |
| $-\sum[y\log\hat{p} + (1-y)\log(1-\hat{p})]$ | Cross-Entropy Loss | What we minimize |
| $Accuracy = \frac{TP+TN}{Total}$ | Accuracy | Overall correctness |
| $Precision = \frac{TP}{TP+FP}$ | Precision | Positive prediction accuracy |
| $Recall = \frac{TP}{TP+FN}$ | Recall | Positive detection rate |
| $F1 = \frac{2PR}{P+R}$ | F1-Score | Harmonic mean |
| $AUC$ | Area Under ROC | Overall model quality |

---
**End of Logistic Regression Tutorial**